# Session 3: a cell that runs down
**A. Real-world problem.** A cell is not a fixed 1.10 V source. Its voltage depends on how
much of each chemical is left (the Nernst equation), it sags when current flows (internal
resistance), it delivers the most power only at one load, and a short circuit turns almost
all of its energy into heat. Today you simulate all four with one small model and check it
against Faraday's law and the Nernst limits.

**Plan.** (1) Nernst voltage, (2) concentrations as charge is drawn, (3) terminal voltage
and the discharge curve, (4) validation, (5) power versus load, temperature and a short circuit.

*Illustrative parameters:* the volume, the internal resistance, the resistance rise as Cu2+
runs out and the thermal numbers below are chosen for teaching, not measured from a real cell.
*Safety: real lithium and lead-acid batteries can be dangerous. Educational model only.*

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

R_GAS = 8.314462618        # J/(mol K)
FARADAY = 96485.0          # C/mol
T0 = 298.15                # K (25 C)
N_E = 2                    # electrons per Zn2+/Cu2+ reaction
E0 = 1.10                  # V, standard Daniell cell voltage
M_ZN = 65.38               # g/mol

# ILLUSTRATIVE cell parameters (not measured)
VOL = 0.020                # L of solution in each half cell
C_CU0 = 1.0                # mol/L Cu2+ at the start
C_ZN0 = 0.1                # mol/L Zn2+ at the start
R_INT0 = 0.8               # ohm, internal resistance at 25 C
K_DEP = 0.05               # resistance rises by K_DEP * C_CU0 / c_Cu as Cu2+ runs out
EA = 20000.0               # J/mol, "activation" energy for how resistance changes with temperature
C_FLOOR = 1e-9            # mol/L, keeps the logarithm finite
Q_MAX = N_E * FARADAY * VOL * C_CU0       # C: charge the Cu2+ in solution can supply

## Step 1: the Nernst equation
For the reaction Zn + Cu2+ -> Zn2+ + Cu the reaction quotient is Q = [Zn2+] / [Cu2+]
(concentrations in mol/L; pure metals count as 1). The cell voltage is

    E = E0 - (R T / (n F)) ln Q

* R T / F at 25 C is 25.69 mV: every factor e (2.718) in Q changes E by 25.69 mV / n.
* One *decade* (factor 10) in Q changes E by 2.303 R T / (n F): 59.16 mV for n = 1, and
  29.58 mV for the Daniell cell (n = 2).

In [ ]:
def area(y, x):
    """Trapezoid-rule integral of y dx (works on every NumPy version)."""
    y, x = np.asarray(y, dtype=float), np.asarray(x, dtype=float)
    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))


def thermal_voltage(T=T0, n=1):
    """R T / (n F) in volts."""
    return R_GAS * T / (n * FARADAY)


def nernst(Q, T=T0, e0=E0, n=N_E):
    """Cell voltage in volts for reaction quotient Q (Q > 0)."""
    Q = np.asarray(Q, dtype=float)
    if np.any(Q <= 0):
        raise ValueError("reaction quotient must be positive")
    return e0 - thermal_voltage(T, n) * np.log(Q)


def mv_per_decade(T=T0, n=1):
    return 1000.0 * thermal_voltage(T, n) * np.log(10.0)

## Step 2: concentrations while the cell discharges
Drawing a charge q (coulombs) reacts xi = q / (n F) moles. Each litre of Cu2+ solution loses
xi / VOL mol and the zinc solution gains the same.

In [ ]:
def concentrations(q):
    xi = np.asarray(q, dtype=float) / (N_E * FARADAY)
    c_zn = C_ZN0 + xi / VOL
    c_cu = np.maximum(C_CU0 - xi / VOL, C_FLOOR)
    return c_zn, c_cu


def ocv(q, T=T0):
    """Open-circuit voltage after charge q has been drawn."""
    c_zn, c_cu = concentrations(q)
    return nernst(c_zn / c_cu, T)

## Step 3: terminal voltage and the discharge curve
With a load resistance `R` the current is I = E / (R + r) and the voltage you can use is
V = E - I r (the terminal voltage). The internal resistance `r` is larger when cold
(an Arrhenius-type factor, illustrative) and rises as Cu2+ runs out (illustrative).

In [ ]:
def r_internal(q, T=T0, r0=R_INT0):
    _, c_cu = concentrations(q)
    temp_factor = np.exp(EA / R_GAS * (1.0 / T - 1.0 / T0))
    return r0 * temp_factor * (1.0 + K_DEP * C_CU0 / c_cu)


def current(q, R, T=T0, r0=R_INT0):
    return ocv(q, T) / (R + r_internal(q, T, r0))


def terminal_voltage(q, R, T=T0, r0=R_INT0):
    return ocv(q, T) - current(q, R, T, r0) * r_internal(q, T, r0)


def discharge_curve(R, T=T0, r0=R_INT0, n=3001):
    """Quasi-static discharge in the charge domain. Returns arrays q, e, i, v and time t."""
    u = np.linspace(0.0, 1.0, n)
    q = Q_MAX * (1.0 - 1e-4) * (1.0 - (1.0 - u) ** 4)          # denser near the end
    e, i, v = ocv(q, T), current(q, R, T, r0), terminal_voltage(q, R, T, r0)
    t = np.concatenate([[0.0], np.cumsum(0.5 * (1 / i[1:] + 1 / i[:-1]) * np.diff(q))])
    return {"q": q, "e": e, "i": i, "v": v, "t": t, "r": r_internal(q, T, r0)}


def usable_capacity_ah(R, T=T0, r0=R_INT0, drop=0.10):
    """Charge drawn (Ah) before the terminal voltage has fallen by `drop` from its starting value."""
    c = discharge_curve(R, T, r0)
    target = (1.0 - drop) * c["v"][0]
    idx = np.nonzero(c["v"] < target)[0]
    if len(idx) == 0:
        return float(c["q"][-1] / 3600.0)
    j = idx[0]
    q_cut = np.interp(target, [c["v"][j], c["v"][j - 1]], [c["q"][j], c["q"][j - 1]])
    return float(q_cut / 3600.0)


def energy_wh(R, T=T0, r0=R_INT0, drop=0.10):
    """Energy delivered to the load (Wh) until the cutoff defined by `drop`."""
    c = discharge_curve(R, T, r0, n=6001)
    target = (1.0 - drop) * c["v"][0]
    m = c["v"] >= target
    q, v = c["q"][m], c["v"][m]
    return float(area(v, q) / 3600.0)

## Step 4: validation against Faraday and Nernst
1. **Time-domain solver.** Solve dq/dt = I(q) with fourth-order Runge-Kutta and compare
   with the charge-domain result (time = integral of dq / I).
2. **Faraday's law.** The zinc dissolved, found from the concentration change, equals
   Q M / (n F) from the charge that flowed.
3. **Energy balance.** chemical energy integral of E dq = delivered integral of V dq + heat integral of I r dq.

In [ ]:
def rk4_charge(R, t_end, steps=2000, T=T0, r0=R_INT0):
    """Charge (C) drawn after t_end seconds, from dq/dt = I(q) solved with RK4."""
    h = t_end / steps
    q = 0.0
    for _ in range(steps):
        k1 = current(q, R, T, r0)
        k2 = current(q + 0.5 * h * k1, R, T, r0)
        k3 = current(q + 0.5 * h * k2, R, T, r0)
        k4 = current(q + h * k3, R, T, r0)
        q += h * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
    return float(q)


def zinc_from_concentration(q):
    """Zinc dissolved (g), found from how much Zn2+ the solution gained."""
    c_zn, _ = concentrations(q)
    return float((c_zn - C_ZN0) * VOL * M_ZN)


def zinc_from_faraday(q):
    return float(q * M_ZN / (N_E * FARADAY))


def energy_balance(R, T=T0, r0=R_INT0):
    """(chemical, delivered, heat) in joules over the whole discharge curve."""
    c = discharge_curve(R, T, r0, n=6001)
    return (float(area(c["e"], c["q"])), float(area(c["v"], c["q"])), float(area(c["i"] * c["r"], c["q"])))


def pack_runtime_hours(ns, npar, R_lamp, v_min, T=T0, r0=R_INT0, n=4001):
    """Hours a lamp of resistance R_lamp stays at or above v_min volts when powered by a pack of
    `ns` cells in series per string and `npar` identical strings in parallel (same model per cell)."""
    if ns < 1 or npar < 1:
        raise ValueError("a pack needs at least one cell")
    u = np.linspace(0.0, 1.0, n)
    q = Q_MAX * (1.0 - 1e-4) * (1.0 - (1.0 - u) ** 4)          # charge drawn from each cell
    e, r = ocv(q, T), r_internal(q, T, r0)
    i_total = ns * e / (R_lamp + ns * r / npar)
    v_lamp = i_total * R_lamp
    if v_lamp[0] < v_min:
        return 0.0
    bad = np.nonzero(v_lamp < v_min)[0]
    j = bad[0] if len(bad) else n - 1
    i_string = i_total / npar
    dt = 0.5 * (1 / i_string[1:j + 1] + 1 / i_string[:j]) * np.diff(q[:j + 1])
    return float(np.sum(dt) / 3600.0)

## Step 5: power, temperature and a short circuit
Delivered power is P = I^2 R = E^2 R / (R + r)^2. It is largest when R = r, where
P_max = E^2 / (4 r), and the efficiency R / (R + r) is then only 50 percent: half the chemical
energy is heat inside the cell. A short circuit (R near 0) gives I = E / r and heat I^2 r,
which warms the cell according to a simple lumped model (illustrative numbers):

    C_th dT/dt = I^2 r - hA (T - T_ambient)

In [ ]:
def load_power(E, r, R):
    R = np.asarray(R, dtype=float)
    return E ** 2 * R / (R + r) ** 2


def efficiency(r, R):
    R = np.asarray(R, dtype=float)
    return R / (R + r)


def short_circuit_heating(E, r, C_th, hA, t_end, dt=0.5, T_amb=25.0):
    """Cell temperature (C) versus time for a dead short. Illustrative lumped model (explicit Euler)."""
    if r <= 0 or C_th <= 0:
        raise ValueError("resistance and heat capacity must be positive")
    n = int(round(t_end / dt))
    t = np.arange(n + 1) * dt
    T = np.empty(n + 1)
    T[0] = T_amb
    heat = (E / r) ** 2 * r
    for k in range(n):
        T[k + 1] = T[k] + dt * (heat - hA * (T[k] - T_amb)) / C_th
    return t, T


def short_circuit_exact(E, r, C_th, hA, t, T_amb=25.0):
    """Closed form of the same model, used to check the numerical one."""
    heat = (E / r) ** 2 * r
    return T_amb + heat / hA * (1.0 - np.exp(-np.asarray(t) * hA / C_th))

## Step 6: many cycles, a little capacity lost each time
A rechargeable cell is discharged, recharged to full, discharged again. Each cycle permanently
loses a small fraction `loss` of the capacity that is left (0.2 percent per cycle is typical of
a modest cell; real fade is not constant). Because the loss is a *fraction of what remains*,
the capacity after n cycles is geometric:  f(n) = (1 - loss)^n.  After 50 cycles of 0.2 percent
that is 0.998^50 = 0.9047, i.e. 90.5 percent. (A loss of 0.2 percent of the *original*
capacity per cycle would give exactly 90.0 percent: the two differ because the second takes
the same bite every time.) An aged cell behaves like a smaller cell: the same voltage curve,
stretched shorter along the charge axis by f, so the usable capacity and the energy per
discharge both scale with f.

In [ ]:
LOSS_PER_CYCLE = 0.002


def capacity_fraction(n, loss=LOSS_PER_CYCLE):
    """Capacity left after n cycles as a fraction of the new cell: (1 - loss)^n."""
    return (1.0 - loss) ** np.asarray(n, dtype=float)


def cycles_to_fraction(target=0.8, loss=LOSS_PER_CYCLE):
    """Whole cycles after which the capacity first falls to `target` or below (end of life at 0.8)."""
    if not 0.0 < loss < 1.0 or not 0.0 < target < 1.0:
        raise ValueError("loss and target must lie strictly between 0 and 1")
    return int(np.ceil(np.log(target) / np.log(1.0 - loss) - 1e-12))


def cycle_life(R, n_cycles, loss=LOSS_PER_CYCLE, T=T0, r0=R_INT0, drop=0.10):
    """Simulate n_cycles of discharge (to the `drop` cutoff) and full recharge, cycle by cycle.

    Returns arrays: cycle number 1..n, capacity fraction at the start of each cycle, usable
    capacity (Ah) and energy (Wh) delivered in that cycle. The fresh-cell numbers are computed once."""
    ah0, wh0 = usable_capacity_ah(R, T, r0, drop), energy_wh(R, T, r0, drop)
    frac = 1.0
    cyc, fr, ah, wh = [], [], [], []
    for k in range(1, int(n_cycles) + 1):
        cyc.append(k)
        fr.append(frac)
        ah.append(ah0 * frac)
        wh.append(wh0 * frac)
        frac *= (1.0 - loss)             # recharge restores the starting concentrations, minus the permanent loss
    return {"cycle": np.array(cyc), "fraction": np.array(fr), "ah": np.array(ah), "wh": np.array(wh)}

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print(f"RT/F at 25 C = {1000 * thermal_voltage():.3f} mV per e-fold (n = 1)")
    print(f"decade, n = 1: {mv_per_decade(T0, 1):.2f} mV;  n = 2 (Daniell): {mv_per_decade(T0, 2):.2f} mV")
    print(f"E at Q = 0.1: {float(nernst(0.1)):.4f} V;  at Q = 1: {float(nernst(1.0)):.4f} V;  at Q = 10: {float(nernst(10.0)):.4f} V")
    print(f"capacity from Cu2+: {Q_MAX:.0f} C = {Q_MAX / 3600:.3f} Ah")

    # figure 1: Nernst sweep
    logq = np.linspace(-4, 4, 161)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(logq, nernst(10.0 ** logq), color="#2a6fdb", label="25 C")
    ax.plot(logq, nernst(10.0 ** logq, T=273.15), "--", color="#2a9d8f", label="0 C")
    ax.plot(logq, nernst(10.0 ** logq, T=323.15), ":", color="#d9534f", label="50 C")
    ax.axhline(E0, color="gray", lw=0.8)
    ax.set_xlabel("log10 of Q = [Zn2+] / [Cu2+]")
    ax.set_ylabel("Daniell cell voltage E (V)")
    ax.set_title(f"Nernst: a straight line, {mv_per_decade(T0, 2):.1f} mV per decade at 25 C", fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "nernst_sweep.png", dpi=120)

    # figure 2: discharge curves
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    for R, col in ((2.0, "#d9534f"), (10.0, "#2a6fdb"), (40.0, "#2a9d8f")):
        c = discharge_curve(R)
        ax2.plot(c["q"] / 3600.0, c["v"], color=col, label=f"load {R:g} ohm: {usable_capacity_ah(R):.2f} Ah to -10%")
    ax2.plot(c["q"] / 3600.0, c["e"], "k:", lw=1, label="open-circuit voltage E")
    ax2.set_xlabel("charge drawn (Ah)")
    ax2.set_ylabel("terminal voltage V = E - I r (V)")
    ax2.set_ylim(0.0, 1.25)
    ax2.set_title("Discharge curves (illustrative parameters)", fontsize=10)
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8, loc="lower left")
    fig2.tight_layout()
    fig2.savefig(OUT / "discharge_curve.png", dpi=120)

    # figure 3: power and efficiency versus load
    Rs = np.linspace(0.05, 6.0, 300)
    fig3, ax3 = plt.subplots(figsize=(6.4, 4.2))
    ax3.plot(Rs, load_power(E0, R_INT0, Rs), color="#2a6fdb", label="power to the load (W)")
    ax3.set_xlabel("load resistance R (ohm)")
    ax3.set_ylabel("power (W)", color="#2a6fdb")
    ax3b = ax3.twinx()
    ax3b.plot(Rs, 100 * efficiency(R_INT0, Rs), color="#d9534f", label="efficiency (%)")
    ax3b.set_ylabel("efficiency (%)", color="#d9534f")
    ax3.axvline(R_INT0, color="gray", ls="--")
    ax3.set_title(f"Maximum power at R = r = {R_INT0} ohm: {float(load_power(E0, R_INT0, R_INT0)):.3f} W at 50 %", fontsize=10)
    ax3.grid(alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(OUT / "power_vs_load.png", dpi=120)

    # figure 4: short circuit
    fig4, ax4 = plt.subplots(figsize=(6.4, 4.2))
    for r, col in ((0.8, "#2a9d8f"), (0.05, "#d9534f")):
        t, T = short_circuit_heating(E0, r, C_th=40.0, hA=0.15, t_end=300.0)
        ax4.plot(t, T, color=col, label=f"r = {r} ohm: I = {E0 / r:.1f} A, heat {E0 ** 2 / r:.1f} W")
        print(f"short circuit r = {r}: I = {E0 / r:.2f} A, heat {E0 ** 2 / r:.2f} W, temperature after 60 s = {T[120]:.1f} C, after 300 s = {T[-1]:.1f} C")
    ax4.set_xlabel("time after the short (s)")
    ax4.set_ylabel("cell temperature (C)")
    ax4.set_title("Short circuit: lumped heating (illustrative numbers)", fontsize=10)
    ax4.grid(alpha=0.3)
    ax4.legend(fontsize=8)
    fig4.tight_layout()
    fig4.savefig(OUT / "short_circuit.png", dpi=120)

    # figure 5: capacity over many cycles
    ncyc = np.arange(0, 201)
    fig5, ax5 = plt.subplots(figsize=(6.4, 4.2))
    ax5.plot(ncyc, 100 * capacity_fraction(ncyc), color="#003f7f", lw=2, label="0.2 % of what is left per cycle")
    ax5.plot(ncyc, 100 * (1 - LOSS_PER_CYCLE * ncyc), "--", color="#c86a00", label="0.2 % of the original per cycle")
    ax5.axhline(80, color="gray", ls=":")
    ax5.plot([50], [100 * capacity_fraction(50)], "o", color="#d81b60")
    ax5.annotate(f"{100 * capacity_fraction(50):.1f} % after 50 cycles", (50, 100 * capacity_fraction(50)), (62, 93), fontsize=9)
    n80 = cycles_to_fraction(0.8)
    ax5.annotate(f"80 % after {n80} cycles", (n80, 80), (n80 + 8, 82), fontsize=9)
    ax5.set_xlabel("cycle number")
    ax5.set_ylabel("capacity (% of a new cell)")
    ax5.set_title("Capacity fade: geometric loss against straight-line loss", fontsize=10)
    ax5.set_ylim(60, 101)
    ax5.grid(alpha=0.3)
    ax5.legend(fontsize=8, loc="lower left")
    fig5.tight_layout()
    fig5.savefig(OUT / "capacity_fade.png", dpi=120)
    cl = cycle_life(10.0, n80)
    print(f"cycles to 80 %: {n80};  after 50 cycles: {100 * capacity_fraction(50):.2f} %;  10 ohm load: {cl['ah'][0]:.3f} Ah at cycle 1, {cl['ah'][49]:.3f} Ah at cycle 50;  energy over life to 80 %: {cl['wh'].sum():.1f} Wh")

    # numbers quoted in the lesson text
    for R in (2.0, 10.0, 40.0):
        c = discharge_curve(R)
        print(f"load {R:g} ohm: start V = {c['v'][0]:.3f} V, I = {c['i'][0]:.3f} A, usable {usable_capacity_ah(R):.3f} Ah, energy {energy_wh(R):.3f} Wh, time {c['t'][-1] / 3600:.2f} h")
    print(f"P_max = {float(load_power(E0, R_INT0, R_INT0)):.4f} W at R = {R_INT0}")
    for Tc in (0, 25, 45):
        print(f"{Tc} C: r(0) = {float(r_internal(0.0, Tc + 273.15)):.3f} ohm, V_term at 10 ohm = {float(terminal_voltage(0.0, 10.0, Tc + 273.15)):.3f} V")

## Where this stops
Real cells have diffusion limits, side reactions, electrode surface effects and ageing that
this model leaves out. The shape of a discharge curve differs between chemistries. Use
this to understand, never to design or test a real battery.

## Coding tasks
1. **Easy.** Run `nernst` for Q = 0.001, 0.01, 0.1, 1, 10 and print the differences between neighbours; confirm each is 29.58 mV and explain why.
2. **Stretch.** Use `usable_capacity_ah` and `energy_wh` for loads of 5, 20 and 80 ohm. Which load wastes the most as heat? Use `energy_balance` to check your answer.
3. **Explain.** Explain, using the heat equation, why the 0.05 ohm cell reaches 57.5 C after 60 s of a short while the 0.8 ohm cell reaches only 27.0 C.

## Assessment
- **A. Conceptual.** Why is the power to the load greatest when R equals the internal resistance, yet the efficiency is then only 50 percent? What would you choose for a torch you want to run a long time?
- **B. Numerical.** For E = 1.10 V and r = 0.8 ohm find the load for maximum power, the current, the load power, and the heat made inside the cell at that point.
- **C. Graph and simulation interpretation.** In the discharge curves for loads of 2, 10 and 40 ohm, which curve starts highest and which gives the most usable capacity (to a 10 percent drop in terminal voltage)? Explain the difference using V = E - I r.
- **D. Design and optimisation challenge.** Design a short-circuit safety limit: a cell has E = 1.10 V and r = 0.05 ohm. A short gives I = 22 A and 24.2 W. Find the series resistance that limits the heat made inside the cell to 2 W, and say why this is not a substitute for a proper fuse.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # Nernst limits
    checks.append((f"25 C: RT/F = 25.69 mV per e-fold (got {1000 * L.thermal_voltage():.2f})", abs(1000 * L.thermal_voltage() - 25.69) < 0.01))
    checks.append((f"25 C: 59.16 mV per decade for n = 1 (got {L.mv_per_decade(L.T0, 1):.2f}), 29.58 for n = 2",
                   abs(L.mv_per_decade(L.T0, 1) - 59.16) < 0.01 and abs(L.mv_per_decade(L.T0, 2) - 29.58) < 0.01))
    checks.append(("Nernst gives E0 exactly when Q = 1", abs(float(L.nernst(1.0)) - 1.10) < 1e-12))
    steps = L.nernst(np.array([0.01, 0.1, 1.0, 10.0, 100.0]))
    checks.append(("each decade of Q lowers E by the same 29.58 mV", np.allclose(-np.diff(steps) * 1000, L.mv_per_decade(L.T0, 2), atol=1e-6)))
    # worked example
    e0 = float(L.nernst(0.1))
    r0 = float(L.r_internal(0.0))
    i0 = float(L.current(0.0, 10.0))
    v0 = float(L.terminal_voltage(0.0, 10.0))
    checks.append((f"worked example: [Zn2+] = 0.1, [Cu2+] = 1.0 gives E = 1.1296 V, r = 0.84 ohm, I = 0.1042 A, V = 1.0420 V on 10 ohm (got {e0:.4f}, {r0:.2f}, {i0:.4f}, {v0:.4f})",
                   abs(e0 - 1.1296) < 5e-5 and abs(r0 - 0.84) < 1e-9 and abs(i0 - 0.1042) < 5e-5 and abs(v0 - 1.0420) < 5e-5))
    checks.append(("worked example: load power I^2 R = 0.1086 W and heat in the cell I^2 r = 0.0091 W",
                   abs(i0 ** 2 * 10 - 0.1086) < 5e-5 and abs(i0 ** 2 * r0 - 0.0091) < 5e-5))
    # concentrations conserve moles
    c_zn, c_cu = L.concentrations(1000.0)
    checks.append(("moles are conserved: Zn2+ gained equals Cu2+ lost", abs((c_zn - L.C_ZN0) - (L.C_CU0 - c_cu)) < 1e-12))
    # Faraday's law
    z1, z2 = L.zinc_from_concentration(1000.0), L.zinc_from_faraday(1000.0)
    checks.append((f"Faraday: zinc from concentrations = Q M / (n F) = 0.339 g for 1000 C (got {z1:.4f} and {z2:.4f})",
                   abs(z1 - z2) < 1e-12 and abs(z2 - 0.3388) < 5e-4))
    # RK4 against charge-domain integral
    c = L.discharge_curve(10.0)
    t_ref = float(np.interp(2000.0, c["q"], c["t"]))
    q_rk = L.rk4_charge(10.0, t_ref, steps=2000)
    checks.append((f"RK4 in time reaches the same charge as the integral of dq/I (2000 C vs {q_rk:.2f} C)", abs(q_rk - 2000.0) / 2000.0 < 1e-3))
    q_coarse = L.rk4_charge(10.0, t_ref, steps=200)
    checks.append(("RK4 result is step-size independent (200 versus 2000 steps)", abs(q_coarse - q_rk) / q_rk < 1e-6))
    # energy balance
    chem, deliv, heat = L.energy_balance(10.0)
    checks.append(("energy balance: chemical = delivered + heat", abs(chem - deliv - heat) / chem < 1e-9))
    checks.append(("a lighter load wastes less as heat (heat share 2 ohm > 40 ohm)",
                   (lambda a, b: a[2] / a[0] > b[2] / b[0])(L.energy_balance(2.0), L.energy_balance(40.0))))
    # discharge curve shape
    checks.append(("terminal voltage is below the open-circuit voltage and falls monotonically",
                   bool(np.all(c["v"] < c["e"]) and np.all(np.diff(c["v"]) <= 1e-12))))
    cap = [L.usable_capacity_ah(R) for R in (2.0, 10.0, 40.0)]
    checks.append((f"heavier loads give less usable capacity: {cap[0]:.3f} < {cap[1]:.3f} < {cap[2]:.3f} Ah, all below {L.Q_MAX / 3600:.3f} Ah",
                   cap[0] < cap[1] < cap[2] < L.Q_MAX / 3600.0))
    # maximum power
    Rs = np.linspace(0.05, 6.0, 1000)
    P = L.load_power(1.10, 0.8, Rs)
    checks.append(("maximum power is at R = r (0.8 ohm), P_max = E^2 / (4 r) = 0.378 W",
                   abs(Rs[np.argmax(P)] - 0.8) < 0.01 and abs(P.max() - 1.10 ** 2 / (4 * 0.8)) < 1e-4))
    checks.append(("hand calculation: at maximum power I = E / (2 r) = 0.6875 A", abs(1.10 / (2 * 0.8) - 0.6875) < 1e-12))
    rs = np.sqrt(2.0 / 0.05)
    checks.append((f"design: limiting the heat in a 0.05 ohm cell to 2 W needs I <= 6.32 A, R_s = 0.124 ohm (got {1.10 / rs - 0.05:.3f})",
                   abs(rs - 6.3246) < 1e-3 and abs(1.10 / rs - 0.05 - 0.1239) < 5e-4))
    checks.append(("Section F: the 40 ohm load starts at 1.106 V, the 2 ohm load at 0.795 V",
                   abs(float(L.terminal_voltage(0.0, 40.0)) - 1.106) < 5e-4 and abs(float(L.terminal_voltage(0.0, 2.0)) - 0.795) < 5e-4))
    tt, TT = L.short_circuit_heating(1.10, 0.05, 40.0, 0.15, 60.0)
    t8, T8 = L.short_circuit_heating(1.10, 0.8, 40.0, 0.15, 60.0)
    checks.append((f"short circuit after 60 s: 57.5 C for 0.05 ohm and 27.0 C for 0.8 ohm (got {TT[-1]:.1f}, {T8[-1]:.1f})", abs(TT[-1] - 57.5) < 0.05 and abs(T8[-1] - 27.0) < 0.05))
    checks.append(("efficiency is 50 percent at maximum power", abs(float(L.efficiency(0.8, 0.8)) - 0.5) < 1e-12))
    # temperature
    vs = [float(L.terminal_voltage(0.0, 10.0, T)) for T in (273.15, 298.15, 318.15)]
    checks.append(("cold cells sag more: V at 0 C < 25 C < 45 C", vs[0] < vs[1] < vs[2]))
    # failure case
    t, T = L.short_circuit_heating(1.10, 0.05, 40.0, 0.15, 300.0)
    exact = L.short_circuit_exact(1.10, 0.05, 40.0, 0.15, t)
    checks.append((f"short circuit: numerical heating matches the closed form (largest gap {np.abs(T - exact).max():.2f} K)", np.abs(T - exact).max() < 0.5))
    checks.append(("short circuit current is E / r = 22 A and heat 24.2 W", abs(1.10 / 0.05 - 22.0) < 1e-9 and abs(1.10 ** 2 / 0.05 - 24.2) < 1e-9))
    checks.append(("a dead short (R = 0) still gives a finite current E / r", abs(float(L.current(0.0, 0.0)) - float(L.ocv(0.0)) / 0.84) < 1e-12))
    bad = 0
    for f in (lambda: L.nernst(0.0), lambda: L.short_circuit_heating(1.1, 0.0, 40.0, 0.15, 10.0), lambda: L.pack_runtime_hours(0, 1, 30.0, 9.0)):
        try:
            f()
        except ValueError:
            bad += 1
    checks.append(("invalid inputs (Q = 0, r = 0, empty pack) are rejected", bad == 3))
    # mini-project reference
    best = None
    for ns in range(5, 16):
        for npar in range(1, 6):
            if L.pack_runtime_hours(ns, npar, 30.0, 9.0) >= 5.0 and (best is None or ns * npar < best[0]):
                best = (ns * npar, ns, npar)
    checks.append((f"mini-project reference: smallest pack for a 30 ohm lamp at 9 V or more for 5 h is 10 x 2 = 20 cells (got {best})", best == (20, 10, 2)))
    # ---- many cycles, capacity fade -------------------------------------------------------------
    f50 = float(L.capacity_fraction(50))
    checks.append((f"0.2 percent loss per cycle gives 90.5 percent capacity after 50 cycles (got {100 * f50:.2f} %); a straight-line loss would give 90.0 %",
                   abs(100 * f50 - 90.5) < 0.05 and abs(1 - 0.002 * 50 - 0.90) < 1e-12 and f50 > 0.90))
    n80 = L.cycles_to_fraction(0.8)
    checks.append((f"end of life (80 percent) comes after {n80} cycles: 0.998^111 > 0.8 >= 0.998^112",
                   n80 == 112 and float(L.capacity_fraction(111)) > 0.8 >= float(L.capacity_fraction(112))))
    cl = L.cycle_life(10.0, n80)
    ah0, wh0 = L.usable_capacity_ah(10.0), L.energy_wh(10.0)
    checks.append(("the cycle-by-cycle simulation follows (1 - loss)^(k - 1) and cycle 1 is the fresh cell",
                   np.allclose(cl["fraction"], 0.998 ** np.arange(n80)) and abs(cl["ah"][0] - ah0) < 1e-12 and abs(cl["wh"][0] - wh0) < 1e-12))
    checks.append(("energy over the whole life to 80 percent equals the geometric series wh0 (1 - 0.998^n) / 0.002",
                   abs(cl["wh"].sum() - wh0 * (1 - 0.998 ** n80) / 0.002) < 1e-9))
    z = L.cycle_life(10.0, 5, loss=0.0)
    checks.append(("zero loss: every cycle is identical", np.allclose(z["ah"], ah0) and np.allclose(z["fraction"], 1.0)))

    def aged_capacity(f, R=10.0):
        """Independent check: an aged cell is the same cell with a solution volume f times smaller."""
        q = np.linspace(0.0, L.Q_MAX * f * (1 - 1e-4), 20001)
        xi = q / (L.N_E * L.FARADAY)
        cz, cc = L.C_ZN0 + xi / (f * L.VOL), np.maximum(L.C_CU0 - xi / (f * L.VOL), 1e-9)
        e = L.nernst(cz / cc)
        r = L.R_INT0 * (1 + L.K_DEP * L.C_CU0 / cc)
        v = e - e / (R + r) * r
        j = np.nonzero(v < 0.9 * v[0])[0][0]
        return np.interp(0.9 * v[0], [v[j], v[j - 1]], [q[j], q[j - 1]]) / 3600.0
    checks.append(("an aged cell behaves like a smaller cell: usable capacity of the 90.5 percent cell matches the independent small-volume calculation to 0.1 percent",
                   abs(aged_capacity(f50) - ah0 * f50) / (ah0 * f50) < 1e-3))
    bad = 0
    for args in ((0.8, -0.1), (0.8, 1.0), (1.2, 0.002)):
        try:
            L.cycles_to_fraction(*args)
        except ValueError:
            bad += 1
    checks.append(("a negative loss, a loss of 100 percent and an end-of-life target above 1 are refused", bad == 3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
